In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
# 모든 라이브러리를 설치 후에 세션을 다시 시작해주세요
# Colab 셀에 한 번만 실행
!pip install -q protobuf==3.20.3 gensim==4.3.1
!pip -q uninstall -y scipy
!pip -q install scipy==1.9.3
!pip -q install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 11.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.4/33.4 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages tha

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 시각화
import matplotlib.pyplot as plt

# 텍스트 데이터 전처리
import re
# import contractions
from collections import Counter
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize

# Tensorflow 라이브러리
import tensorflow as tf

# Tensorflow 전처리
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# word2vec 모델
from gensim.models import KeyedVectors
from gensim.models import Word2Vec

# Tensorflow 모델 구조
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import MultiHeadAttention, Conv1D, Bidirectional, LSTM, GlobalAveragePooling1D, Lambda, Attention, ReLU,Softmax
from tensorflow.keras.layers import Input, Embedding, Flatten, Layer, Multiply, Concatenate, Dropout, Dense, Reshape, Permute, Dot
from tensorflow.keras.utils import plot_model

# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils


In [ ]:
# 텍스트 패딩을 생성한 후, 리스트의 형태로 DataFrame에 저장
def create_pad_sequence(df, col, max_sequence_length = 10):

    # 토큰화 및 패딩
    text_sequence = tokenizer.texts_to_sequences(df[col])
    padding_result = pad_sequences(text_sequence, maxlen=max_sequence_length, padding='post')
    padding_result # 새로 만들어준 사전에서의 인덱스

    # 'padded_sequences' 열을 df에 추가하고, 패딩이 적용된 시퀀스의 내용을 할당
    df[col + '_padded_sequences'] = list(padding_result)

    # 'padded_sequences' 열의 데이터 타입을 object로 변경하여 각 요소를 리스트 형태로 저장
    df[col + '_padded_sequences'] = df[col + '_padded_sequences'].apply(lambda x: list(x))

# 데이터셋 로딩

In [ ]:
# 데이터 로딩
df = pd.read_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v4.json')

print(df.shape)
df.head()

(66718, 15)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 12.1029911041, 0.0, 15.7332105637, 4.268...","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[-0.2068804801, -0.3263846636, 0.1318071485, -...","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0685713813, 0.1392135769, -0.0395870171, -..."
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 0.0, 2.2863385677, 10.4048433304, 6.9813...","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[0.062450625, -0.0707582533, 0.1557247639, -0....","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0461796485, 0.1261856854, -0.0177110955, -..."
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 37.5705032349, 2.7596354485, 0.0, 12.911...","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[-0.2690293193, -0.42407187820000003, 0.823390...","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0621351376, 0.1333664954, -0.0194056146, -..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 18.6175479889, 0.0, 0.0, 3.5440309048, 0...","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[0.1193233952, -0.5735993385, 0.40760195260000...","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0678028613, 0.051634714000000005, -0.01779..."
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[1.9380488396, 0.0, 0.0, 6.4230856895, 0.0, 0....","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[-0.4373431206, -0.6204138398, 0.1469407529, -...","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0794128701, 0.0517616905, -0.0255422704, -..."


# Text Embedding


In [ ]:
# Path to the Google News Word2Vec model
model_path = '/content/drive/MyDrive/GoogleNews-vectors-negative300.bin'
# Load the model
google_w2v_model = KeyedVectors.load_word2vec_format(model_path, binary=True)
# Print the shape of the model to confirm its size
print('Model shape:', google_w2v_model.vectors.shape)
# Output should be (3000000, 300) -> to do: (3000000개에서 사용되는 단어만 select)

Model shape: (3000000, 300)


In [ ]:
# 최대 사용할 단어 수와 시퀀스의 최대 길이
max_words = 100000

# 토크나이저 초기화 및 학습
tokenizer = Tokenizer(num_words=max_words)

In [ ]:
# lemmatized text 기준으로 사전 만들어주기
keyword = df['text'].explode().unique()
tokenizer.fit_on_texts(keyword)

In [ ]:
# 최종 단어 index의 길이가 max word를 넘는 상황, 65만개 dataset 기준
len(tokenizer.word_index)

47118

In [ ]:
# 단어 인덱스 맵 생성 및 임베딩 행렬 준비
# 임베딩 차원을 300으로 가정
word_index = tokenizer.word_index
total_words = min(max_words, len(word_index))
embedding_matrix = np.zeros((total_words, 300))

# 새로 만든 사전에 맞게 임베딩 해줘야함
# 임베딩 행렬의 크기 출력
for word, i in word_index.items():
    if i > max_words:
        continue
    if word in google_w2v_model:
        embedding_matrix[i] = google_w2v_model[word]

print(embedding_matrix)

[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.08007812  0.10498047  0.04980469 ...  0.00366211  0.04760742
  -0.06884766]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]


In [ ]:
len(embedding_matrix)

47118

# User, Item Embedding을 위한 전처리
- user별, item별 임베딩을 생성한 다음, 전체 DataFrame에 concat할 예정
- 실행 시간 단축을 위한 방안

In [ ]:
total_words

47118

In [ ]:
df['user_id'].nunique()

7449

In [ ]:
df['asin'].nunique()

16099

In [ ]:
# user별, item별로 텍스트를 연결
user_text_df = df.groupby('user_id')['text'].apply(lambda x: ' '.join(x)).reset_index()
item_text_df = df.groupby('asin')['text'].apply(lambda x: ' '.join(x)).reset_index()

# column 변경
user_text_df.columns = ['user_id', 'user_text']
item_text_df.columns = ['asin', 'item_text']

# 결과 확인
print(user_text_df.shape)
print(item_text_df.shape)

user_text_df.head()

(7449, 2)
(16099, 2)


,user_id,user_text
0,AE22ZYH6WIY6WJHJRWOMPQCDJHHA,I ordered these as a gift for a loved one. It ...
1,AE23NZUELB4BYWLHKWJXAS73PTSQ,These packing cubes are very nice quality. The...
2,AE23ZBUF2YVBQPH2NN6F5XSA3QYQ,The vacuum sealed snowsuit came and while I wa...
3,AE24VPMWEEQD62YPOG53BW7JCGFA,This is a very nice set of heavy-duty workboot...
4,AE25JLQJ6LWKPHI3JFHBN5F4YN2A,"Unfortunately my dish arrived damaged, which i..."


# Create Embedding Matrix

In [ ]:
# sequence 길이 500으로 embedding 생성
create_pad_sequence(user_text_df,'user_text', max_sequence_length = 500)
create_pad_sequence(item_text_df,'item_text', max_sequence_length = 500)

# Data Concat

In [ ]:
# 원래의 데이터셋에 결합해서 최종 데이터셋을 만들어줌

df = pd.merge(df, user_text_df, on='user_id', how='left')
df = pd.merge(df, item_text_df, on='asin', how='left')

In [ ]:
# 최종 결과에 이상이 있는지 확인
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66718 entries, 0 to 66717
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   user_id                     66718 non-null  object
 1   asin                        66718 non-null  object
 2   parent_asin                 66718 non-null  object
 3   text                        66718 non-null  object
 4   rating                      66718 non-null  int64 
 5   title_y                     66718 non-null  object
 6   roberta_emb                 66718 non-null  object
 7   small_image_url             66718 non-null  object
 8   medium_image_url            66718 non-null  object
 9   large_image_url             66718 non-null  object
 10  VGG16_emb                   66718 non-null  object
 11  user_bert_emb               66718 non-null  object
 12  item_bert_emb               66718 non-null  object
 13  user_roberta_emb            66718 non-null  ob

# User, Item label encoding
- 라벨 인코딩
- train, test 분리



In [ ]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

In [ ]:
# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

# train과 test data를 분리
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, valid = train_test_split(train, test_size=0.125, random_state=42)

train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [ ]:
# inputs
user_text_train = np.array(train['user_text_padded_sequences'].tolist(),dtype='int64')
user_text_valid = np.array(valid['user_text_padded_sequences'].tolist(),dtype='int64')
user_text_test = np.array(test['user_text_padded_sequences'].tolist(),dtype='int64')

item_text_train = np.array(train['item_text_padded_sequences'].tolist(),dtype='int64')
item_text_valid = np.array(valid['item_text_padded_sequences'].tolist(),dtype='int64')
item_text_test = np.array(test['item_text_padded_sequences'].tolist(),dtype='int64')

# rating
y_train = train['rating'].to_numpy()
y_valid = valid['rating'].to_numpy()
y_test = test['rating'].to_numpy()

# ANR

In [ ]:
def build_aspect_based_neural_recommender(embedding_matrix, vocab_size=total_words, embedding_dim=300, projection_dim=50, num_aspects=5, sequence_length=500, learning_rate=0.002):

    # 사용자와 아이템 입력 정의
    user_input = Input(shape=(sequence_length,), name='user_input')  # 사용자 문서 입력
    item_input = Input(shape=(sequence_length,), name='item_input')  # 아이템 문서 입력

    # 임베딩 레이어 정의
    embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False, name='embedding_layer')
    user_embeddings = embedding_layer(user_input)  # Shape: (batch_size, sequence_length, embedding_dim)
    item_embeddings = embedding_layer(item_input)  # Shape: (batch_size, sequence_length, embedding_dim)

    user_embeddings = Reshape((300, 500))(user_embeddings)
    # Dense 레이어를 통해 500 차원을 10으로 줄임
    user_embeddings = Dense(10, activation='relu')(user_embeddings)
    # 다시 원래 차원으로 Reshape 가능
    user_embeddings = Reshape((10, 300))(user_embeddings)

    item_embeddings = Reshape((300, 500))(item_embeddings)
    # Dense 레이어를 통해 500 차원을 10으로 줄임
    item_embeddings = Dense(10, activation='relu')(item_embeddings)
    # 다시 원래 차원으로 Reshape 가능
    item_embeddings = Reshape((10, 300))(item_embeddings)

    # 커스텀 레이어 정의: Aspect-specific projections
    class AspectProjectionLayer(Layer):
        def __init__(self, num_aspects, projection_dim, **kwargs):
            super(AspectProjectionLayer, self).__init__(**kwargs)
            self.num_aspects = num_aspects
            self.projection_dim = projection_dim
            self.projection_layers = [
                Dense(projection_dim, name=f'aspect_projection_{i}') for i in range(num_aspects)
            ]

        def call(self, inputs):
            projected_embeddings = [proj(inputs) for proj in self.projection_layers]
            # Stack along a new axis to combine the aspect projections
            stacked_embeddings = tf.stack(projected_embeddings, axis=1)
            return stacked_embeddings

    class AspectAttentionLayer(Layer):
        def __init__(self, context_window_size, projection_dim, **kwargs):
            super(AspectAttentionLayer, self).__init__(**kwargs)
            self.context_window_size = context_window_size
            self.projection_dim = projection_dim
            self.concat = Concatenate(axis=-1)
            self.attention_dense = Dense(1)
            self.softmax = Softmax(axis=1)

        def call(self, inputs):
            # inputs: (batch_size, sequence_length, projection_dim)
            sequence_length = tf.shape(inputs)[1]
            half_window = self.context_window_size // 2

            # Generate context windows for each word
            padded_inputs = Lambda(lambda x: tf.pad(x, [[0, 0], [half_window, half_window], [0, 0]]))(inputs)
            context_windows = []
            for i in range(self.context_window_size):
                context_slice = Lambda(lambda x: x[:, i:i+sequence_length, :])(padded_inputs)
                context_windows.append(context_slice)

            # Concatenate context windows: (batch_size, sequence_length, context_window_size * projection_dim)
            context_embeddings = self.concat(context_windows)

            # Compute attention scores
            attention_scores = self.attention_dense(context_embeddings)  # (batch_size, sequence_length, 1)
            attention_scores = self.softmax(attention_scores)  # Apply softmax to get the attention distribution

            # Weighted sum of aspect-specific word representations
            weighted_sum = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=1))([attention_scores, inputs])  # (batch_size, projection_dim)
            return weighted_sum

    # 사용자와 아이템 임베딩에 커스텀 레이어 적용
    user_aspect_embeddings = AspectProjectionLayer(num_aspects, projection_dim, name='user_aspect_embeddings')(user_embeddings)
    item_aspect_embeddings = AspectProjectionLayer(num_aspects, projection_dim, name='item_aspect_embeddings')(item_embeddings)

    user_aspect_attention_outputs = []
    item_aspect_attention_outputs = []
    # 사용자 임베딩에 대한 attention 적용
    for i in range(num_aspects):
        # 각 aspect 차원을 따로 추출 (batch_size, sequence_length, projection_dim)
        user_aspect_input = Lambda(lambda x: x[:, i, :, :])(user_aspect_embeddings)
        item_aspect_input = Lambda(lambda x: x[:, i, :, :])(item_aspect_embeddings)

        # AspectAttentionLayer 인스턴스를 적용
        user_aspect_output = AspectAttentionLayer(3, projection_dim)(user_aspect_input)
        item_aspect_output = AspectAttentionLayer(3, projection_dim)(item_aspect_input)

        # 결과를 리스트에 추가
        user_aspect_attention_outputs.append(user_aspect_output)
        item_aspect_attention_outputs.append(item_aspect_output)

    # 리스트를 다시 텐서로 변환 (batch_size, num_aspects, projection_dim)
    user_aspect_representation = Lambda(lambda x: tf.stack(x, axis=1))(user_aspect_attention_outputs)
    item_aspect_representation = Lambda(lambda x: tf.stack(x, axis=1))(item_aspect_attention_outputs)

    # Co-Attention 적용
    class CoAttentionLayer(Layer):
        def __init__(self, projection_dim, **kwargs):
            super(CoAttentionLayer, self).__init__(**kwargs)
            self.projection_dim = projection_dim
            self.softmax = Softmax(axis=1)

        def call(self, user_rep, item_rep):
            # Transpose item representation for dot product
            item_rep_transposed = Lambda(lambda x: tf.transpose(x, perm=[0, 2, 1]))(item_rep)  # (batch_size, projection_dim, num_aspects)

            # Compute affinity matrix
            affinity_matrix = Lambda(lambda x: tf.matmul(x[0], x[1]))([user_rep, item_rep_transposed])  # (batch_size, num_aspects, num_aspects)

            # Apply softmax to get attention distributions
            user_attention = self.softmax(affinity_matrix)  # (batch_size, num_aspects, num_aspects)
            item_attention = self.softmax(Lambda(lambda x: tf.transpose(x, perm=[0, 2, 1]))(affinity_matrix))  # (batch_size, num_aspects, num_aspects)

            # Weighted sum for co-attended representations
            co_attended_user = Lambda(lambda x: tf.matmul(x[0], x[1]))([user_attention, user_rep])  # (batch_size, num_aspects, projection_dim)
            co_attended_item = Lambda(lambda x: tf.matmul(x[0], x[1]))([item_attention, item_rep])  # (batch_size, num_aspects, projection_dim)

            return co_attended_user, co_attended_item

    # Co-Attention 레이어 적용
    co_attention_layer = CoAttentionLayer(projection_dim, name='co_attention_layer')
    co_attended_user, co_attended_item = co_attention_layer(user_aspect_representation, item_aspect_representation)

    # Co-Attended representation을 기반으로 최종 평점 예측
    user_item_interaction = Multiply()([co_attended_user, co_attended_item])  # (batch_size, num_aspects, projection_dim)

    rating_prediction = Dense(1, activation='linear', name='rating_prediction')(user_item_interaction)  # 최종 평점 예측(측면별)
    rating_prediction = Lambda(lambda x: tf.reduce_sum(x, axis=1))(rating_prediction)

    # 모델 정의
    model = Model(inputs=[user_input, item_input], outputs=rating_prediction, name='AspectBasedNeuralRecommenderWithCoAttention')

    # 모델 컴파일
    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = learning_rate), loss = "mse", metrics = ["mse", "mae"])

    return model

# 모델 생성
model = build_aspect_based_neural_recommender(embedding_matrix,total_words)

# 모델 요약 출력
model.summary()

Model: "AspectBasedNeuralRecommenderWithCoAttention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 500)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 500)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_layer     │ (None, 500, 300)  │ 14,135,400 │ user_input[0][0], │
│ (Embedding)         │                   │            │ item_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 300, 500)  │          0 │ embedding_layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 300, 500)  │          0 │ embedding_layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 300, 10)   │      5,010 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 300, 10)   │      5,010 │ reshape_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 10, 300)   │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 10, 300)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_aspect_embedd… │ (None, 5, 10, 50) │     75,250 │ reshape_1[0][0]   │
│ (AspectProjectionL… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_aspect_embedd… │ (None, 5, 10, 50) │     75,250 │ reshape_3[0][0]   │
│ (AspectProjectionL… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 10, 50)    │          0 │ user_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_22 (Lambda)  │ (None, 10, 50)    │          0 │ user_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_44 (Lambda)  │ (None, 10, 50)    │          0 │ user_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_66 (Lambda)  │ (None, 10, 50)    │          0 │ user_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_88 (Lambda)  │ (None, 10, 50)    │          0 │ user_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 10, 50)    │          0 │ item_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_23 (Lambda)  │ (None, 10, 50)    │          0 │ item_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_45 (Lambda)  │ (None, 10, 50)    │          0 │ item_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_67 (Lambda)  │ (None, 10, 50)    │          0 │ item_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_89 (Lambda)  │ (None, 10, 50)    │          0 │ item_aspect_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aspect_attention_l… │ (None, 50)        │        151 │ lambda[0][0]      │
│ (AspectAttentionLa… │                   │            │                 

 Total params: 14,297,481 (54.54 MB)

 Trainable params: 162,081 (633.13 KB)

 Non-trainable params: 14,135,400 (53.92 MB)

In [ ]:
# Configuration of EarlyStopping
early_stopping = EarlyStopping(
    monitor ='val_loss',  # Monitor model's validation loss
    min_delta = 0.001,     # Minimum change to qualify as an improvement
    patience = 5,         # Stop after 5 epochs without improvement
    verbose = 1,           # Print messages
    mode = 'min',          # Stop training when the quantity monitored has stopped decreasing
    restore_best_weights = True  # Restore model weights from the epoch with the best value of the monitored quantity
)

In [ ]:
# 모델 학습
history = model.fit([user_text_train, item_text_train], y_train,
                    validation_data = ([user_text_valid, item_text_valid], y_valid),
                    epochs = 100,
                    batch_size = 128,
                    callbacks=[early_stopping])

Epoch 1/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 50s 55ms/step - loss: 2.4508 - mae: 1.0638 - mse: 2.4508 - val_loss: 0.5503 - val_mae: 0.5754 - val_mse: 0.5503
Epoch 2/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.5419 - mae: 0.5488 - mse: 0.5419 - val_loss: 0.5300 - val_mae: 0.5735 - val_mse: 0.5300
Epoch 3/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.5176 - mae: 0.5361 - mse: 0.5176 - val_loss: 0.5195 - val_mae: 0.5698 - val_mse: 0.5195
Epoch 4/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4881 - mae: 0.5163 - mse: 0.4881 - val_loss: 0.5277 - val_mae: 0.5799 - val_mse: 0.5277
Epoch 5/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4531 - mae: 0.4980 - mse: 0.4531 - val_loss: 0.4992 - val_mae: 0.5045 - val_mse: 0.4992
Epoch 6/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4504 - mae: 0.4960 - mse: 0.4504 - val_loss: 0.5021 - val_mae: 0.4987 - val_mse: 0.5021
Epoch 7/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4372 - mae: 0.4877 - mse: 0.4

In [ ]:
# 모델 성능 측정
predicted_ratings = model.predict([user_text_test, item_text_test])

# MAE
MAE = mean_absolute_error(y_test, predicted_ratings)
print(f"MAE: {MAE:.4f}")

# MSE
MSE = mean_squared_error(y_test, predicted_ratings)
print(f"MSE: {MSE:.4f}")

# RMSE
RMSE = np.sqrt(MSE)
print(f"RMSE: {RMSE:.4f}")

# MAPE
MAPE = mean_absolute_percentage_error(y_test, predicted_ratings) * 100
print(f"MAPE: {MAPE:.4f}%")

417/417 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
MAE: 0.5104
MSE: 0.5100
RMSE: 0.7141
MAPE: 15.0763%
